# core

> Cumulative Mojo cell magic for SolveIt

In [ ]:
#| default_exp core

In [ ]:
#| export
import re, subprocess, sys
from pathlib import Path
from IPython.core.magic import register_cell_magic

In [ ]:
#| export
def filter_moso(msgs):
    "Strip %%moso magic lines and concatenate into one Mojo source"
    pat = re.compile(r"^%%moso[^\n]*\n", re.MULTILINE)
    bodies = [pat.sub("", m["content"], count=1) for m in msgs]
    if not bodies: return ""
    *preamble,last = bodies
    indented = "\n".join("    " + line for line in last.splitlines())
    return "\n".join(preamble + [f"def main() raises:\n{indented}"])

In [ ]:
#| export
async def get_moso():
    "Get concatenated Mojo source from all %%moso cells up to the current one"
    from dialoghelper import read_msg, find_msgs
    cur_id = (await read_msg(0))["id"]
    all_moso = await find_msgs(r"^%%moso", msg_type="code", include_output=False)
    upto = []
    for m in all_moso:
        upto.append(m)
        if m["id"] == cur_id: break
    return filter_moso(upto)

In [ ]:
#| export
def setup_moso(host=None, cmd="mojo run {path}", path="/tmp/cell.mojo"):
    "Register the %%moso magic"
    @register_cell_magic
    async def moso(line, cell):
        source = await get_moso()
        run_cmd = cmd.format(path=path)
        if host:
            full_cmd = f"ssh {host} 'cat > {path} && {run_cmd}'"
            result = subprocess.run(full_cmd, shell=True, input=source.encode(), capture_output=True)
        else:
            Path(path).write_text(source)
            result = subprocess.run(run_cmd, shell=True, capture_output=True)
        if result.returncode == 0:
            if result.stdout: print(result.stdout.decode())
        else: print(result.stderr.decode(), file=sys.stderr)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()